# Analysis of stored Monte Carlo results
Simulations run with `python run_designs.py Dx` (fresh process, stored in `results/`). This notebook only reads them; it never runs a design.

In [ ]:
import numpy as np, pandas as pd
import store, mc, plots
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 300)
nm = {'abadie': 'Abadie, plug-in', 'bcch': 'DML-DiD', 'oracle': 'oracle'}

## Health check: failures, infeasible estimators, numerical diagnostics

In [ ]:
for D in [f'D{i}' for i in range(1, 9)]:
    df = store.load(D)
    if df.empty:
        print(D, 'no results'); continue
    print(D, df.groupby('status').size().to_dict(), '| rank-deficient refits:', int(df.filter(like='rankdef').sum().sum()),
          '| |err| > 10:', int((df.err.abs() > 10).sum()), '| commits:', df.commit.nunique())

## D1 Chang's DGP and the trend version

In [ ]:
d1 = store.load('D1')
display(mc.summary(d1, by=('variant', 'K', 'N', 'est')).round(3))
plots.hist(d1.query("N == 200 and (variant == 'trend' or K == 2)"), row='variant', cols=['abadie', 'bcch', 'oracle'], names=nm)

## D2 density path (decay a) and D3 alignment

In [ ]:
d2 = store.load('D2')
s2 = mc.summary(d2, by=('est', 'a', 'cos', 'N'))
display(s2.loc['bcch'].round(3), mc.slopes(s2.query('N >= 500').loc[['bcch']]).round(2))
plots.path(s2.loc['bcch'].reset_index(), y='bias_sd', hue='a')

In [ ]:
d3 = store.load('D3')
s3 = mc.summary(d3, by=('est', 'a', 'cos', 'N'))
display(s3.round(3))
plots.path(s3.loc['bcch'].reset_index(), y='bias', hue='cos', panel='a')

## D4 which nuisance is dense

In [ ]:
d4 = store.load('D4')
display(mc.summary(d4, by=('dense_g', 'dense_l', 'est')).round(3))

## D5 reference category

In [ ]:
d5 = store.load('D5')
display(mc.summary(d5, by=('rule', 'hit', 'N', 'est', 'basis')).round(3),
        mc.ranges(d5, by=('rule', 'hit', 'N')).round(2),
        mc.diag_quantiles(d5, by=('rule', 'N'), cols=('w_max', 'cond_l', 'rankdef_l')).round(1))
plots.bases(d5.query("rule == 'growing' and hit == 'both' and N == 1000"), order=['A0', 'r0', 'r1', 'r2', 'bad', 'full'])

## D6 overlap

In [ ]:
d6 = store.load('D6')
display(mc.summary(d6[d6.basis.isin(['A0', 'bad'])], by=('scale', 'basis')).round(3), mc.ranges(d6, by=('scale',)).round(2))

## D7 centring

In [ ]:
d7 = store.load('D7')
display(mc.summary(d7, by=('N', 'est', 'basis')).round(3), mc.ranges(d7, by=('N',)).round(2))
plots.bases(d7[d7.N == 1000])

## D8 Sequeira, repeated cross sections

In [ ]:
d8 = store.load('D8')
display(mc.summary(d8, by=('est', 'basis')).round(3), mc.ranges(d8).round(2))

## Section 6: Chang's Table 2, rebuilt step by step
Chang (2020), DML-DiD (Lasso): −5.222 (2.647). Each row removes one departure from his Theorem 3.1.

In [ ]:
from pathlib import Path
f = sorted(Path('../results/application').glob('decomposition-*.parquet'))[-1]
display(pd.read_parquet(f)[['step', 'description', 'theta', 'se', 'change', 'sd_splits', 'kept']].round(3))